# P&P Analysis v2 (PI0.5)

Reads `results_v2/rollouts_v2.db` from [`test_pi05_jennifer_v2.ipynb`](test_pi05_jennifer_v2.ipynb).

Run **after** test notebook Section 6 completes.

This notebook answers four questions from the controlled v2 experiment:

1. **Fixed evaluation slice** — what tasks/episodes were chosen and why?
2. **`multi_sample_select`** — what does it do and how does it compare to other methods?
3. **Action instability metrics** — what do they measure and do they correlate with uncertainty?
4. **Expected findings** — do the DB results support the detector and corrector hypotheses?

Sections A–D compute tables/plots; interpretation cells print results-driven answers from the data.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, sqlite3, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from pathlib import Path

# ── Paths v2 (Drive mount or local copy) ─────────────────────────────────────
SHARED = '/content/drive/MyDrive/cs159-sp26'
V1_DB_PATH = f'{SHARED}/results/rollouts.db'
V1_VIDEO_DIR = f'{SHARED}/results/videos'
RESULTS_DIR = f'{SHARED}/results_v2'
DB_PATH = f'{RESULTS_DIR}/rollouts_v2.db'
VIDEO_DIR = f'{RESULTS_DIR}/videos_v2'
FIGURES_DIR = f'{RESULTS_DIR}/figures_v2'
FINAL_FIGURES_DIR = f'{FIGURES_DIR}/final_v2'

if not os.path.isfile(DB_PATH):
    _local = Path('results_v2/rollouts_v2.db')
    if _local.is_file():
        RESULTS_DIR = str(_local.parent)
        DB_PATH = str(_local)
        VIDEO_DIR = str(_local.parent / 'videos_v2')
        FIGURES_DIR = str(_local.parent / 'figures_v2')
        FINAL_FIGURES_DIR = str(Path(FIGURES_DIR) / 'final_v2')
        print('Using local results:', DB_PATH)

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(FINAL_FIGURES_DIR, exist_ok=True)

def save_fig(name, final=False):
    prefix = 'final_v2_' if final else 'pnp_jennifer_v2_'
    folder = FINAL_FIGURES_DIR if final else FIGURES_DIR
    path = os.path.join(folder, f'{prefix}{name}')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved {path}')

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError(
        f'v2 DB not found at {DB_PATH}. Run test_pi05_jennifer_v2.ipynb Section 6 first.'
    )

con = sqlite3.connect(DB_PATH)
con.row_factory = sqlite3.Row
print('Connected to', DB_PATH)
print('Videos:   ', VIDEO_DIR)
print('Figures:  ', FIGURES_DIR)
print('Final figs:', FINAL_FIGURES_DIR)


In [ ]:
# ── Load tables into DataFrames ───────────────────────────────────────────────
rollouts = pd.read_sql('SELECT * FROM rollouts', con)
steps_df = pd.read_sql('SELECT * FROM pnp_euler_steps', con)

# v1 DB for legacy Phase 1/2/3 exploratory context
rollouts_v1 = steps_v1 = None
if os.path.isfile(V1_DB_PATH):
    _con_v1 = sqlite3.connect(V1_DB_PATH)
    rollouts_v1 = pd.read_sql('SELECT * FROM rollouts', _con_v1)
    steps_v1 = pd.read_sql('SELECT * FROM pnp_euler_steps', _con_v1)
    _con_v1.close()
    print(f'Legacy v1 DB: {V1_DB_PATH}  ({len(rollouts_v1)} rollouts)')
else:
    print(f'Legacy v1 DB not found at {V1_DB_PATH} — legacy stats use v2 non-final rows only')

def parse_step_indices(s):
    try:
        return json.loads(s)
    except Exception:
        return s
rollouts['step_indices_parsed'] = rollouts['pnp_step_indices'].map(parse_step_indices)

# ── Experiment constants (mirror test_pi05_jennifer_v2.ipynb Section 6) ─────────
FINAL_SUITES = ['libero_goal', 'libero_spatial']
FINAL_MAX_TASKS = 8
FINAL_EPISODE_IDXS = list(range(10))
BASELINE_STEPS = 10
EXTRA_STEPS_MATCHED = 16
MULTI_SAMPLE_N = 3
MULTI_SAMPLE_PROBE_STEPS = (2, 3)
FINAL_STEP_CONFIGS = [(2, 3), (3, 4), (4, 5)]

METHOD_ORDER = ['vanilla', 'extra_steps', 'multi_sample_select',
                'pnp_uncertainty_only', 'pnp_refinement']
INSTABILITY_COLS = ['action_delta_l2_mean', 'action_delta_l2_max', 'action_var_mean',
                    'gripper_flip_count', 'gripper_flip_rate', 'chunk_disagreement_mean']
EPISODE_KEYS = ['suite', 'task_idx', 'episode_idx', 'init_state_hash']

def collapse_step_configs(df, success_agg='max'):
    """One row per (episode, method); P&P methods may have multiple step configs per episode."""
    if df.empty:
        return df
    agg = {'success': success_agg}
    for col in ['u_mean_episode', 'n_steps'] + INSTABILITY_COLS:
        if col in df.columns:
            agg[col] = 'mean'
    return (
        df.groupby(['method'] + EPISODE_KEYS, as_index=False)
        .agg(agg)
    )

# ── Legacy exploratory split (v1 DB preferred) ────────────────────────────────
STEP_CONFIG_FILTER = None
if rollouts_v1 is not None:
    legacy_rollouts = rollouts_v1.copy()
    steps_legacy = steps_v1.copy()
    legacy_video_dir = V1_VIDEO_DIR
else:
    if 'final_eval_slice' in rollouts.columns:
        legacy_rollouts = rollouts[rollouts['final_eval_slice'].fillna(0) != 1].copy()
    else:
        legacy_rollouts = rollouts.copy()
    steps_legacy = steps_df.copy()
    legacy_video_dir = VIDEO_DIR

pnp_all  = legacy_rollouts[legacy_rollouts.pnp_enabled == 1].copy()
baseline = legacy_rollouts[legacy_rollouts.pnp_enabled == 0].copy()
phase2 = pnp_all[pnp_all['pnp_mode'] == 'uncertainty'].copy()
phase3 = pnp_all[pnp_all['pnp_mode'] == 'both'].copy()
all_step_configs = sorted(pnp_all['pnp_step_indices'].dropna().unique().tolist())

print(f'rollouts total: {len(rollouts)} rows')
print(f'  legacy rows: {len(legacy_rollouts)}')
print(f'  Phase 2 (uncertainty): {len(phase2)}  |  Phase 3 (refine): {len(phase3)}')

# ── v2 final experiment rows ────────────────────────────────────────────────────
final_df = rollouts[rollouts['final_eval_slice'].fillna(0) == 1].copy() if 'final_eval_slice' in rollouts.columns else pd.DataFrame()
final_episode_df = collapse_step_configs(final_df, success_agg='max') if not final_df.empty else pd.DataFrame()
method_summary = recovery_df = det_metrics_df = taxonomy_df = pd.DataFrame()

if final_df.empty:
    print('  final_eval_slice=1: 0 rows — run test_pi05_jennifer_v2.ipynb Section 6 first.')
else:
    print(f'  final_eval_slice=1: {len(final_df)} rows')
    if 'method' in final_df.columns:
        print('  Methods:', final_df['method'].value_counts().to_dict())


---
## Section 0 — Experiment design

### Fixed evaluation slice

Instead of reporting all LIBERO results from v1 (4 suites × 10 tasks × 10 episodes), v2 uses a **fixed, paired slice** so every method runs on the same episodes:

| Parameter | Value |
|---|---|
| Suites | `libero_goal`, `libero_spatial` |
| Tasks | Top **8** by v1 Phase-1 vanilla failure count in those suites |
| Episodes | Indices **0–9** per task |
| Init states | LIBERO fixed init states — identical across all methods |

**Why this slice?**
- **Paired comparison:** same `(suite, task_idx, episode_idx, init_state_hash)` for every method.
- **Failure-rich tasks:** ranking by v1 failure count picks episodes where P&P has signal to detect and potentially fix.
- **Runtime:** ~80 episodes × 5 methods is feasible; the full v1 sweep is not.
- **Defensibility:** a narrow controlled slice is easier to explain than 400 exploratory rollouts.


### Methods

| Method | What it does | Purpose |
|---|---|---|
| `vanilla` | Standard π0.5, 10 Euler steps, P&P off | Main baseline |
| `extra_steps` | Same policy, **16** Euler steps (matched to 2-step P&P with K=3), no P&P | Matched-compute baseline |
| `multi_sample_select` | Sample **3** action chunks per observation; probe P&P uncertainty at steps `[2,3]`; **pick lowest-U sample** | Sampling baseline using uncertainty for selection |
| `pnp_uncertainty_only` | P&P measure-only at steps `[2,3]`, `[3,4]`, `[4,5]` | Detector analysis |
| `pnp_refinement` | P&P refine + measure at same step configs | Correction test |

#### `multi_sample_select` in detail

π0.5 predicts action **chunks** (multiple timesteps at once). For each new chunk:

1. Draw **3** candidate chunks with different random seeds.
2. For each candidate, run P&P in **uncertainty-only** mode at Euler steps `[2, 3]` and record mean uncertainty `U`.
3. Execute the chunk with the **lowest** `U`.
4. Repeat when the action queue is empty.

This uses P&P uncertainty for **selection**, not trajectory refinement. It tests whether "sample more and pick the most self-consistent action" explains any gains that might otherwise be attributed to P&P refinement.


### Action instability metrics

Logged per rollout to test whether P&P uncertainty tracks **action instability**, not just task outcome:

| Metric | Meaning |
|---|---|
| `action_delta_l2_mean` / `max` | Mean/max L2 distance between **consecutive executed actions** — jerkiness |
| `action_var_mean` | Average variance across the 7 action dimensions over the episode |
| `gripper_flip_count` / `rate` | How often the gripper command crosses open/closed — erratic grasping |
| `chunk_disagreement_mean` | L2 distance between actions at **chunk boundaries** — plan discontinuities |

**Expected pattern:** failures with high uncertainty should also show high instability (jerky motion, gripper flips). Failures with **low** uncertainty and **low** instability likely reflect perception/grounding errors rather than denoising instability.


### Hypotheses (what we look for)

**Detector (Section C):** `pnp_uncertainty_only` episodes with high `u_mean_episode` should fail more often and correlate with instability metrics (Pearson/Spearman, F1, AUC).

**Corrector (Section A):** `pnp_refinement` should beat `vanilla`, `extra_steps`, and `multi_sample_select` on the **same** fixed slice. If `extra_steps` or `multi_sample_select` match refinement, gains may be generic extra compute, not P&P-specific correction.

**Recovery (Section B, secondary):** Among vanilla failures, interventions should recover some episodes — but this is **not** the headline performance claim (use Section A for that).

**Failure taxonomy:** Bucket episodes into high-U/high-instability failures vs low-U/low-instability failures to separate denoising instability from wrong-mode/perception failures.


In [ ]:
def describe_final_slice(df):
    """Print slice inventory from DB and flag missing methods."""
    print('=' * 72)
    print('FIXED EVALUATION SLICE (from rollouts_v2.db)')
    print('=' * 72)
    if df.empty:
        print('No final_eval_slice rows — run test_pi05_jennifer_v2.ipynb Section 6 first.')
        return

    n_episodes = df.groupby(EPISODE_KEYS).ngroups
    tasks = (
        df.groupby(['suite', 'task_idx', 'task_desc'], as_index=False)
        .agg(n_episodes=('episode_idx', 'nunique'))
        .sort_values(['suite', 'task_idx'])
    )
    print(f'Suites: {sorted(df.suite.unique())}')
    print(f'Tasks: {len(tasks)}  |  Unique episodes: {n_episodes}')
    print(f'Expected: up to {FINAL_MAX_TASKS} tasks, {len(FINAL_EPISODE_IDXS)} episodes each')
    print()
    print('Tasks in slice:')
    for _, r in tasks.iterrows():
        desc = str(r['task_desc'])[:55]
        print(f"  {r['suite']:<18} task {r['task_idx']:>2}  ({r['n_episodes']} eps)  {desc}...")

    if 'method' in df.columns:
        print()
        print('Method row counts:')
        for m in METHOD_ORDER:
            n = (df['method'] == m).sum()
            status = 'OK' if n > 0 else 'MISSING'
            print(f'  {m:<24} {n:>5} rows  [{status}]')
        missing = [m for m in METHOD_ORDER if (df['method'] == m).sum() == 0]
        if missing:
            print(f'\nMissing methods: {missing}')
            print('Run test_pi05_jennifer_v2.ipynb Section 6 for extra_steps + multi_sample_select.')

    print()
    print('Selection rationale: top tasks by v1 vanilla failure count in', FINAL_SUITES)
    print('=' * 72)

describe_final_slice(final_df)


---
## Section A — Full matched evaluation

Compare all methods on the **same** fixed episodes. A supported **correction claim** requires `pnp_refinement` success rate to exceed both matched-compute baselines (`extra_steps`, `multi_sample_select`) — not just vanilla.


In [ ]:
if final_df.empty:
    print('No final_eval_slice rows yet — run test_pi05_jennifer_v2.ipynb Section 6 first.')
else:
    final_episode_df = collapse_step_configs(final_df, success_agg='max')
    agg_cols = {
        'success': 'mean', 'u_mean_episode': 'mean', 'n_steps': 'mean',
        **{c: 'mean' for c in INSTABILITY_COLS if c in final_episode_df.columns},
    }
    method_summary = (
        final_episode_df.groupby('method', as_index=False)
        .agg(n_episodes=('success', 'count'), **{k: (k, v) for k, v in agg_cols.items() if k in final_episode_df.columns})
        .rename(columns={'success': 'success_rate'})
    )
    method_summary['method'] = pd.Categorical(method_summary['method'], categories=METHOD_ORDER, ordered=True)
    method_summary = method_summary.sort_values('method')
    print('=== Full matched evaluation ===')
    display(method_summary.round(4))

    fig, ax = plt.subplots(figsize=(8, 4))
    colors = plt.cm.Set2(np.linspace(0, 1, len(method_summary)))
    ax.bar(method_summary['method'].astype(str), method_summary['success_rate'], color=colors)
    ax.set_ylabel('Success rate')
    ax.set_xlabel('Method')
    ax.set_ylim(0, 1)
    ax.set_title('Full matched evaluation: success by method (v2 slice)')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    save_fig('success_by_method.png', final=True)
    plt.show()

    inst_cols = [c for c in ['action_delta_l2_mean', 'gripper_flip_rate'] if c in method_summary.columns]
    if inst_cols:
        fig, axes = plt.subplots(1, len(inst_cols), figsize=(5 * len(inst_cols), 4))
        if len(inst_cols) == 1:
            axes = [axes]
        for ax, col in zip(axes, inst_cols):
            ax.bar(method_summary['method'].astype(str), method_summary[col], color=colors)
            ax.set_title(col)
            ax.set_xlabel('Method')
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right')
        fig.suptitle('Action instability by method (v2 slice)', fontsize=11)
        plt.tight_layout()
        save_fig('instability_by_method.png', final=True)
        plt.show()

    method_summary.to_csv(os.path.join(RESULTS_DIR, 'final_v2_method_comparison.csv'), index=False)
    print(f'Wrote {RESULTS_DIR}/final_v2_method_comparison.csv')


In [ ]:
def interpret_matched_evaluation(summary):
    print('=' * 72)
    print('INTERPRETATION: Full matched evaluation (Section A)')
    print('=' * 72)
    if summary.empty:
        print('No method summary — run Section 6 in test notebook first.')
        return

    sr = dict(zip(summary['method'].astype(str), summary['success_rate']))
    vanilla_sr = sr.get('vanilla')
    refine_sr = sr.get('pnp_refinement')
    extra_sr = sr.get('extra_steps')
    multi_sr = sr.get('multi_sample_select')

    ranked = summary.sort_values('success_rate', ascending=False)
    print('Success rate ranking:')
    for _, r in ranked.iterrows():
        delta = ''
        if vanilla_sr is not None and r['method'] != 'vanilla':
            delta = f"  (Δ vs vanilla: {r['success_rate'] - vanilla_sr:+.1%})"
        print(f"  {r['method']:<24} {r['success_rate']:.1%}{delta}")

    print()
    if refine_sr is not None:
        print(f'pnp_refinement SR: {refine_sr:.1%}')
        if extra_sr is not None:
            diff = refine_sr - extra_sr
            verdict = 'beats' if diff > 0.01 else ('ties' if abs(diff) <= 0.01 else 'loses to')
            print(f'  vs extra_steps ({extra_sr:.1%}): {verdict} extra compute baseline ({diff:+.1%})')
        else:
            print('  vs extra_steps: no data')
        if multi_sr is not None:
            diff = refine_sr - multi_sr
            verdict = 'beats' if diff > 0.01 else ('ties' if abs(diff) <= 0.01 else 'loses to')
            print(f'  vs multi_sample_select ({multi_sr:.1%}): {verdict} sampling baseline ({diff:+.1%})')
            print('  multi_sample_select samples 3 chunks and picks lowest-U — if it ties/beats')
            print('  refinement, gains may be from sampling rather than P&P trajectory correction.')
        else:
            print('  vs multi_sample_select: no data')
        if vanilla_sr is not None and refine_sr > vanilla_sr + 0.01:
            print('  Correction improves over vanilla.')
        elif vanilla_sr is not None:
            print('  Correction does not clearly beat vanilla on this slice.')
    else:
        print('pnp_refinement: no data')

    if multi_sr is not None and vanilla_sr is not None:
        print(f'\nmulti_sample_select alone: {multi_sr:.1%} vs vanilla {vanilla_sr:.1%} ({multi_sr - vanilla_sr:+.1%})')

    for col, label in [('action_delta_l2_mean', 'action jerk'), ('gripper_flip_rate', 'gripper flips')]:
        if col not in summary.columns:
            continue
        v = summary.loc[summary['method'] == 'vanilla', col]
        m = summary.loc[summary['method'] == 'multi_sample_select', col]
        if len(v) and len(m) and pd.notna(v.iloc[0]) and pd.notna(m.iloc[0]):
            change = m.iloc[0] - v.iloc[0]
            direction = 'lower' if change < 0 else 'higher'
            print(f'multi_sample_select {label}: {direction} than vanilla ({change:+.4f})')
    print('=' * 72)

interpret_matched_evaluation(method_summary)


---
## Section B — Recovery-only evaluation

Restrict to episodes where **vanilla failed** and ask how often each intervention recovers them. Also track **degradation** (vanilla succeeded but intervention failed).

> **Note:** This is a secondary "rescue" story. The headline performance claim comes from Section A (full matched SR on all episodes), not recovery rate alone.


In [ ]:
if final_df.empty:
    print('No final_eval_slice data.')
else:
    join_keys = ['suite', 'task_idx', 'episode_idx', 'init_state_hash']
    vanilla = final_df[final_df['method'] == 'vanilla'][join_keys + ['success']].rename(
        columns={'success': 'vanilla_success'})
    episode_df = collapse_step_configs(final_df, success_agg='max')
    merged = episode_df.merge(vanilla, on=join_keys, how='inner')
    vanilla_fail = merged[merged['vanilla_success'] == 0].copy()

    recovery_rows = []
    for method in METHOD_ORDER:
        if method == 'vanilla':
            continue
        sub = vanilla_fail[vanilla_fail['method'] == method]
        if sub.empty:
            continue
        degradation_sub = merged[(merged['method'] == method) & (merged['vanilla_success'] == 1)]
        recovery_rows.append(dict(
            method=method, n_vanilla_failures=len(sub),
            recovery_rate=sub['success'].mean(),
            degradation_rate=(1 - degradation_sub['success']).mean() if len(degradation_sub) else np.nan,
        ))

    recovery_df = pd.DataFrame(recovery_rows)
    if not recovery_df.empty:
        recovery_df['method'] = pd.Categorical(recovery_df['method'], categories=METHOD_ORDER, ordered=True)
        recovery_df = recovery_df.sort_values('method')
        print('=== Recovery-only (vanilla failures) ===')
        display(recovery_df.round(4))

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(recovery_df['method'].astype(str), recovery_df['recovery_rate'], color='steelblue')
        ax.set_ylabel('Recovery rate')
        ax.set_xlabel('Method')
        ax.set_ylim(0, 1)
        ax.set_title('Recovery rate on vanilla failures (v2 slice)')
        plt.xticks(rotation=20, ha='right')
        plt.tight_layout()
        save_fig('recovery_by_method.png', final=True)
        plt.show()

        recovery_df.to_csv(os.path.join(RESULTS_DIR, 'final_v2_recovery_metrics.csv'), index=False)
        print(f'Wrote {RESULTS_DIR}/final_v2_recovery_metrics.csv')
    else:
        recovery_df = pd.DataFrame()
        print('No recovery data — need vanilla + intervention rows on same episodes.')


In [ ]:
def interpret_recovery(recovery):
    print('=' * 72)
    print('INTERPRETATION: Recovery-only evaluation (Section B)')
    print('=' * 72)
    if recovery.empty:
        print('No recovery data available.')
        return
    best = recovery.loc[recovery['recovery_rate'].idxmax()]
    print(f'Highest recovery rate: {best["method"]} at {best["recovery_rate"]:.1%}')
    print(f'  (on {int(best["n_vanilla_failures"])} vanilla failures)')
    print()
    for _, r in recovery.iterrows():
        deg = r['degradation_rate']
        deg_str = f'{deg:.1%}' if pd.notna(deg) else 'N/A'
        print(f"  {r['method']:<24} recovery={r['recovery_rate']:.1%}  degradation={deg_str}")
    print()
    print('Use Section B for the rescue story; use Section A for overall performance claims.')
    print('=' * 72)

interpret_recovery(recovery_df)


---
## Section C — Detector analysis

Test whether `pnp_uncertainty_only` uncertainty predicts failure and correlates with action instability.


In [ ]:
try:
    from sklearn.metrics import roc_auc_score, average_precision_score
    _HAS_SKLEARN = True
except ImportError:
    _HAS_SKLEARN = False

def threshold_curve(df):
    umin, umax = df.u_mean_episode.min(), df.u_mean_episode.max()
    thresholds = np.linspace(umin, umax, 100) if umin < umax else np.array([umin])
    precisions, recalls, f1s = [], [], []
    y_true = (1 - df['success']).astype(int)
    for t in thresholds:
        y_pred = (df.u_mean_episode >= t).astype(int)
        tp = ((y_pred == 1) & (y_true == 1)).sum()
        fp = ((y_pred == 1) & (y_true == 0)).sum()
        fn = ((y_pred == 0) & (y_true == 1)).sum()
        prec = tp / (tp + fp) if (tp + fp) else 0
        rec  = tp / (tp + fn) if (tp + fn) else 0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
        precisions.append(prec); recalls.append(rec); f1s.append(f1)
    best_idx = int(np.argmax(f1s))
    return thresholds, precisions, recalls, f1s, dict(
        threshold=thresholds[best_idx], precision=precisions[best_idx],
        recall=recalls[best_idx], f1=f1s[best_idx])

det_df = collapse_step_configs(
    final_df[final_df['method'] == 'pnp_uncertainty_only'].dropna(subset=['u_mean_episode']),
    success_agg='max',
) if not final_df.empty else pd.DataFrame()
detector_metrics = []

if det_df.empty:
    print('No pnp_uncertainty_only rows in final slice.')
    det_metrics_df = pd.DataFrame()
else:
    det_raw = final_df[final_df['method'] == 'pnp_uncertainty_only'].dropna(subset=['u_mean_episode'])
    configs = sorted(det_raw['pnp_step_indices'].dropna().unique())
    fig, axes = plt.subplots(1, max(1, len(configs)), figsize=(12, 4))
    if len(configs) == 1:
        axes = [axes]
    for ax, cfg in zip(axes, configs):
        sub = det_raw[det_raw['pnp_step_indices'] == cfg]
        succ = sub[sub.success == 1]['u_mean_episode']
        fail = sub[sub.success == 0]['u_mean_episode']
        ax.violinplot([succ, fail], positions=[1, 2], showmeans=True)
        ax.set_xticks([1, 2]); ax.set_xticklabels(['success', 'failure'])
        ax.set_title(f'steps={cfg}')
        ax.set_ylabel('Mean episode uncertainty')
    fig.suptitle('P&P uncertainty: success vs failure (v2 detector)', fontsize=11)
    plt.tight_layout()
    save_fig('uncertainty_success_failure.png', final=True)
    plt.show()

    for cfg in configs:
        sub = det_raw[det_raw['pnp_step_indices'] == cfg]
        if len(sub) < 3:
            continue
        r_pearson, p_pearson = stats.pearsonr(sub['u_mean_episode'], sub['success'])
        r_spearman, p_spearman = stats.spearmanr(sub['u_mean_episode'], sub['success'])
        _, _, _, _, best = threshold_curve(sub)
        row = dict(step_config=cfg, n=len(sub),
                   pearson_r=r_pearson, pearson_p=p_pearson,
                   spearman_r=r_spearman, spearman_p=p_spearman,
                   best_f1=best['f1'], best_threshold=best['threshold'],
                   best_precision=best['precision'], best_recall=best['recall'])
        if _HAS_SKLEARN and sub['success'].nunique() > 1:
            y_true = (1 - sub['success']).astype(int)
            row['roc_auc'] = roc_auc_score(y_true, sub['u_mean_episode'])
            row['pr_auc'] = average_precision_score(y_true, sub['u_mean_episode'])
        for ic in ['action_delta_l2_mean', 'gripper_flip_rate']:
            if ic in sub.columns and sub[ic].notna().any():
                r_i, p_i = stats.spearmanr(sub['u_mean_episode'], sub[ic])
                row[f'spearman_u_vs_{ic}'] = r_i
        detector_metrics.append(row)

    det_metrics_df = pd.DataFrame(detector_metrics)
    print('=== Detector metrics ===')
    display(det_metrics_df.round(4))
    det_metrics_df.to_csv(os.path.join(RESULTS_DIR, 'final_v2_detector_metrics.csv'), index=False)

    if 'action_delta_l2_mean' in det_df.columns:
        fig, ax = plt.subplots(figsize=(6, 5))
        sc = ax.scatter(det_df['u_mean_episode'], det_df['action_delta_l2_mean'],
                        c=det_df['success'], cmap='RdYlGn', alpha=0.7)
        ax.set_xlabel('Mean episode uncertainty')
        ax.set_ylabel('Action delta L2 mean')
        ax.set_title('Uncertainty vs action instability (v2)')
        plt.colorbar(sc, label='success')
        plt.tight_layout()
        save_fig('uncertainty_vs_instability.png', final=True)
        plt.show()

    cfg_summary = det_raw.groupby('pnp_step_indices').agg(
        success_rate=('success', 'mean'), u_mean=('u_mean_episode', 'mean'), n=('rollout_id', 'count'),
    ).reset_index()
    fig, ax1 = plt.subplots(figsize=(7, 4))
    x = np.arange(len(cfg_summary))
    ax1.bar(x - 0.2, cfg_summary['success_rate'], width=0.4, label='SR')
    ax2 = ax1.twinx()
    ax2.bar(x + 0.2, cfg_summary['u_mean'], width=0.4, color='orange', alpha=0.7, label='mean U')
    ax1.set_xticks(x); ax1.set_xticklabels(cfg_summary['pnp_step_indices'], rotation=15)
    ax1.set_ylabel('Success rate'); ax2.set_ylabel('Mean uncertainty')
    ax1.set_title('Detector: step config comparison (v2)')
    plt.tight_layout()
    save_fig('step_config_comparison.png', final=True)
    plt.show()


In [ ]:
def classify_failure_modes(episode_df, u_col='u_mean_episode', inst_col='action_delta_l2_mean'):
    """Bucket episodes into failure taxonomy cells."""
    if episode_df.empty or u_col not in episode_df.columns:
        print('Insufficient data for failure taxonomy.')
        return pd.DataFrame()

    det = episode_df[episode_df['method'] == 'pnp_uncertainty_only'].copy()
    if det.empty:
        print('No pnp_uncertainty_only rows for taxonomy.')
        return pd.DataFrame()

    u_med = det[u_col].median()
    inst_med = det[inst_col].median() if inst_col in det.columns and det[inst_col].notna().any() else None

    def _bucket(row):
        high_u = row[u_col] >= u_med
        if inst_med is not None and inst_col in row and pd.notna(row[inst_col]):
            high_inst = row[inst_col] >= inst_med
        else:
            high_inst = None
        if row['success'] == 1:
            return 'high_U_success' if high_u else 'low_U_success'
        if high_inst is None:
            return 'high_U_failure' if high_u else 'low_U_failure'
        if high_u and high_inst:
            return 'high_U_high_instability_failure'
        if not high_u and not high_inst:
            return 'low_U_low_instability_failure'
        if high_u:
            return 'high_U_low_instability_failure'
        return 'low_U_high_instability_failure'

    det['taxonomy'] = det.apply(_bucket, axis=1)
    counts = det['taxonomy'].value_counts().reset_index()
    counts.columns = ['taxonomy', 'count']

    print('=== Failure taxonomy (pnp_uncertainty_only) ===')
    display(counts)
    for tax in counts['taxonomy']:
        sub = det[det['taxonomy'] == tax].head(2)
        examples = [f"({r['suite']}, T{r['task_idx']}, ep{r['episode_idx']})" for _, r in sub.iterrows()]
        if examples:
            print(f'  {tax}: e.g. {", ".join(examples)}')

    out_path = os.path.join(RESULTS_DIR, 'final_v2_failure_taxonomy.csv')
    det[['suite', 'task_idx', 'episode_idx', 'init_state_hash', 'success',
         u_col, inst_col if inst_col in det.columns else u_col, 'taxonomy']].to_csv(out_path, index=False)
    print(f'Wrote {out_path}')
    return det

def interpret_detector(metrics_df):
    print('=' * 72)
    print('INTERPRETATION: Detector analysis (Section C)')
    print('=' * 72)
    if metrics_df.empty:
        print('No detector metrics available.')
        return
    for _, r in metrics_df.iterrows():
        cfg = r['step_config']
        print(f"\nStep config {cfg} (n={int(r['n'])}):")
        sp = r.get('spearman_r', np.nan)
        if pd.notna(sp):
            direction = 'positive' if sp > 0 else 'negative'
            support = 'supports' if sp > 0.1 else ('weakly supports' if sp > 0 else 'does not support')
            print(f'  Spearman(U, success)={sp:.3f} — {support} detector ({direction} correlation)')
        if pd.notna(r.get('best_f1')):
            print(f"  Best F1={r['best_f1']:.3f} at threshold={r['best_threshold']:.4f}")
        if pd.notna(r.get('roc_auc')):
            print(f"  ROC-AUC={r['roc_auc']:.3f}")
        for ic in ['action_delta_l2_mean', 'gripper_flip_rate']:
            key = f'spearman_u_vs_{ic}'
            if key in r and pd.notna(r[key]):
                link = 'supports instability link' if abs(r[key]) > 0.2 else 'weak/no instability link'
                print(f'  Spearman(U, {ic})={r[key]:.3f} — {link}')
    print('=' * 72)

taxonomy_df = classify_failure_modes(
    collapse_step_configs(final_df, success_agg='max') if not final_df.empty else pd.DataFrame()
)
interpret_detector(det_metrics_df)


---
## Section D — Findings from this run

Data-driven summary combining slice inventory, method comparison, detector metrics, and failure taxonomy.


In [ ]:
def summarize_v2_findings(final_df, method_summary, recovery_df, det_metrics_df, taxonomy_df):
    print('=' * 72)
    print('V2 FINDINGS SUMMARY (data-driven)')
    print('=' * 72)

    # 1. Slice
    if final_df.empty:
        print('\n1. SLICE: No final_eval_slice data yet.')
    else:
        n_tasks = final_df.groupby(['suite', 'task_idx']).ngroups
        n_eps = final_df.groupby(EPISODE_KEYS).ngroups
        suites = ', '.join(sorted(final_df.suite.unique()))
        print(f'\n1. SLICE: {n_eps} unique episodes across {n_tasks} tasks in [{suites}].')
        print('   Selected as top failure-count tasks from v1; episodes 0-9; same inits across methods.')

    # 2. multi_sample_select
    print('\n2. MULTI_SAMPLE_SELECT:')
    print(f'   Samples {MULTI_SAMPLE_N} chunks per observation, probes U at steps {MULTI_SAMPLE_PROBE_STEPS}, picks lowest-U.')
    if not method_summary.empty and 'multi_sample_select' in method_summary['method'].astype(str).values:
        ms = method_summary[method_summary['method'] == 'multi_sample_select'].iloc[0]
        van = method_summary[method_summary['method'] == 'vanilla']
        van_sr = van.iloc[0]['success_rate'] if len(van) else None
        print(f'   SR = {ms["success_rate"]:.1%}', end='')
        if van_sr is not None:
            print(f' (vs vanilla {van_sr:.1%}, Δ={ms["success_rate"] - van_sr:+.1%})')
        else:
            print()
        refine = method_summary[method_summary['method'] == 'pnp_refinement']
        if len(refine):
            ref_sr = refine.iloc[0]['success_rate']
            diff = ref_sr - ms['success_rate']
            if abs(diff) <= 0.01:
                print('   Ties pnp_refinement — sampling may explain correction gains.')
            elif diff > 0:
                print(f'   pnp_refinement beats multi_sample_select by {diff:+.1%} — P&P correction adds value beyond selection.')
            else:
                print(f'   multi_sample_select beats pnp_refinement by {-diff:+.1%} — selection baseline is stronger.')
    else:
        print('   No multi_sample_select data in DB yet.')

    # 3. Instability
    print('\n3. ACTION INSTABILITY METRICS:')
    if not det_metrics_df.empty and 'spearman_u_vs_action_delta_l2_mean' in det_metrics_df.columns:
        vals = det_metrics_df['spearman_u_vs_action_delta_l2_mean'].dropna()
        if len(vals):
            mean_r = vals.mean()
            link = 'Uncertainty correlates with action jerk' if abs(mean_r) > 0.2 else 'Weak/no correlation between U and action jerk'
            print(f'   Mean Spearman(U, action_delta_l2_mean) = {mean_r:.3f} — {link}.')
    elif not method_summary.empty and 'action_delta_l2_mean' in method_summary.columns:
        print('   Instability logged per method; see Section A instability bars and Section C scatter.')
    else:
        print('   Instability columns not yet populated in DB.')

    # 4. Hypotheses
    print('\n4. HYPOTHESIS CHECK:')
    if not det_metrics_df.empty:
        sp_mean = det_metrics_df['spearman_r'].mean() if 'spearman_r' in det_metrics_df.columns else np.nan
        f1_mean = det_metrics_df['best_f1'].mean() if 'best_f1' in det_metrics_df.columns else np.nan
        det_ok = pd.notna(sp_mean) and sp_mean > 0.1
        print(f'   Detector: {"SUPPORTED" if det_ok else "NOT CLEARLY SUPPORTED"} (mean Spearman r={sp_mean:.3f}, mean F1={f1_mean:.3f})')
    else:
        print('   Detector: insufficient data.')

    if not method_summary.empty:
        sr = dict(zip(method_summary['method'].astype(str), method_summary['success_rate']))
        refine_sr = sr.get('pnp_refinement')
        baselines = [sr.get('extra_steps'), sr.get('multi_sample_select'), sr.get('vanilla')]
        baselines = [b for b in baselines if b is not None]
        if refine_sr is not None and baselines:
            beats_all = all(refine_sr > b + 0.01 for b in baselines)
            beats_vanilla = sr.get('vanilla') is not None and refine_sr > sr['vanilla'] + 0.01
            if beats_all:
                print(f'   Corrector: SUPPORTED — pnp_refinement ({refine_sr:.1%}) beats all baselines.')
            elif beats_vanilla:
                print(f'   Corrector: PARTIALLY SUPPORTED — beats vanilla ({sr["vanilla"]:.1%}) but not all matched baselines.')
            else:
                print(f'   Corrector: NOT SUPPORTED — pnp_refinement ({refine_sr:.1%}) does not clearly beat baselines.')
        else:
            print('   Corrector: insufficient data.')
    else:
        print('   Corrector: insufficient data.')

    if not taxonomy_df.empty:
        hi_hi = (taxonomy_df['taxonomy'] == 'high_U_high_instability_failure').sum()
        lo_lo = (taxonomy_df['taxonomy'] == 'low_U_low_instability_failure').sum()
        print(f'   Taxonomy: {hi_hi} high-U/high-instability failures, {lo_lo} low-U/low-instability failures.')
        print('   (High-U + fail → likely denoising instability; low-U + fail → likely perception/grounding.)')

    if not recovery_df.empty:
        best_rec = recovery_df.loc[recovery_df['recovery_rate'].idxmax()]
        print(f'\n   Recovery (secondary): best={best_rec["method"]} at {best_rec["recovery_rate"]:.1%} on vanilla failures.')

    print('\n5. CAVEATS:')
    print('   - Use Section A for performance claims; Section B for rescue story only.')
    print('   - π0.5 remains main model; SmolVLA is a stretch reproducibility follow-up.')
    print('=' * 72)

if not final_df.empty:
    final_episode_inst = collapse_step_configs(final_df, success_agg='max')
    inst_summary = (
        final_episode_inst.groupby('method', as_index=False)
        .agg(**{c: (c, 'mean') for c in INSTABILITY_COLS if c in final_episode_inst.columns},
             n_episodes=('success', 'count'))
    )
    inst_summary.to_csv(os.path.join(RESULTS_DIR, 'final_v2_instability_metrics.csv'), index=False)
    print('=== Instability summary ===')
    display(inst_summary.round(4))
    print(f'Wrote {RESULTS_DIR}/final_v2_instability_metrics.csv')

summarize_v2_findings(final_df, method_summary, recovery_df, det_metrics_df, taxonomy_df)
